# Assignment 5: Data Visualization using matplotlib - AQI trends

**Dataset:** `aqi.csv`. On Colab keep it in `MyDrive/Data-Science`; on local Jupyter keep it next to this notebook.

In [ ]:
# Mount Google Drive (run in Google Colab)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CSV_PATH = '/content/drive/MyDrive/Data-Science/aqi.csv'
except ImportError:  # local Jupyter
    CSV_PATH = 'aqi.csv'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())
display(df.dtypes.to_frame('data_type'))

## Find the date, AQI, and pollutant columns

The notebook guesses common column names. Review the detected mapping and edit values if your CSV uses different names.

In [ ]:
def find_col(names, keywords):
    for col in names:
        low = col.lower().replace(' ', '').replace('_', '')
        if any(word in low for word in keywords):
            return col
    return None

DATE_COL = find_col(df.columns, ['date','datetime','timestamp','time'])
AQI_COL = find_col(df.columns, ['aqi','airqualityindex'])
# Exclude the date and AQI fields from pollutant detection.
POLLUTANT_COLS = [c for c in df.columns if c not in [DATE_COL, AQI_COL] and find_col([c], ['pm25','pm10','no2','so2','o3','pollutant'])]

print('Detected date column:', DATE_COL)
print('Detected AQI column:', AQI_COL)
print('Detected pollutant columns:', POLLUTANT_COLS)
# If detection is incorrect, set explicitly, for example:
# DATE_COL = 'Date'
# AQI_COL = 'AQI'
# POLLUTANT_COLS = ['PM2.5', 'PM10', 'CO']

## Prepare the data

In [ ]:
if DATE_COL:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce', dayfirst=True)
    df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL)
for col in ([AQI_COL] if AQI_COL else []) + POLLUTANT_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Rows after date cleanup:', len(df))
print('Missing numeric values:')
display(df[[c for c in ([AQI_COL] if AQI_COL else []) + POLLUTANT_COLS]].isna().sum().to_frame('missing_count'))

## Plot the overall AQI trend

In [ ]:
if not DATE_COL or not AQI_COL:
    print('A date or AQI column was not detected. Set DATE_COL and AQI_COL in the mapping cell above.')
else:
    plt.figure(figsize=(12, 5))
    plt.plot(df[DATE_COL], df[AQI_COL], color='darkorange', linewidth=1.5, label='AQI')
    plt.title('AQI over time')
    plt.xlabel('Date')
    plt.ylabel('AQI')
    plt.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()